# Order Nakeds

In [ ]:
## THIS CELL SHOULD BE IN ALL VSCODE NOTEBOOKS ##

MARKET = "NSE"

# Set the root
from from_root import from_root # type: ignore
ROOT = from_root()

import pandas as pd # type: ignore
from loguru import logger # type: ignore

pd.options.display.max_columns = None
pd.set_option('display.precision', 2)

from pathlib import Path
import sys

# Add `src` and ROOT to _src.pth in .venv to allow imports in VS Code
from sysconfig import get_path

if "src" not in Path.cwd().parts:
    src_path = str(Path(get_path("purelib")) / "_src.pth")
    with open(src_path, "w") as f:
        f.write(str(ROOT / "src\n"))
        f.write(str(ROOT))
        if str(ROOT) not in sys.path:
            sys.path.insert(1, str(ROOT))

# Start the Jupyter loop
from ib_async import util # type: ignore

util.startLoop()

logger.add(sink=ROOT / "log" / "ztest.log", mode="w")

## Imports

In [ ]:
from utils import handle_nse_raws, get_pickle, load_config, arrange_orders, pickle_me, get_file_age, yes_or_no
from datetime import datetime
from ib_async import IB
from ibfuncs import get_open_orders, quick_pf, place_orders, make_ib_orders

## Set constants

In [ ]:
config = load_config(MARKET)
port = config.get('PORT')
MARGINPERORDER = config.get('MARGINPERORDER')

## Handle Raws

In [ ]:
# Consolidate raw files to df_nakeds.pkl
pattern = str(f"*{MARKET.lower()}nakeds*.pkl")

handle_nse_raws(pattern=pattern)

In [ ]:
file_path = ROOT / 'data' / 'nse_nakeds.pkl'

def how_many_days_old(file_path) -> float:
    """Gets the file's age in days"""
    file_age = get_file_age(file_path=file_path)
    
    seconds_in_a_day = 86400
    file_age_in_days = file_age.td.total_seconds() / seconds_in_a_day if file_age else 0
    
    return file_age_in_days

In [ ]:
## Check the age of df_pickles, before ordering
txt = f"df_nakeds.pkl is {how_many_days_old(file_path): 0.2f}. Want to load?"
ans = yes_or_no(txt)

if ans:
    df_opts = get_pickle(file_path)
    print('\n\n')
    print(df_opts.drop(columns=['nse_symbol', 'instrument', 'contract', 'expiry']).head())
else:
    print('Bye!!!')

## check open orders

In [ ]:
# check open orders
with IB().connect(port=port, clientId=10) as ib:
    dfo = get_open_orders(ib)
    dfp = quick_pf(ib)
    

In [ ]:
if not dfo.empty:
    remove_opens = set(dfo.symbol.to_list())
else:
    remove_opens = set()

In [ ]:
# make a list of symbols to be removed from df_opts

if not dfp.empty:
    remove_positions = set(dfp.symbol.to_list())
else:
    remove_positions = set()

remove_ib_syms = remove_opens | remove_positions

# get the target options to plant
dft = df_opts[~df_opts.ib_symbol.isin(remove_ib_syms)].reset_index(drop=True)

In [ ]:
len(dft)

## Arrange and make orders

In [ ]:
df_nakeds = arrange_orders(dft, maxmargin=MARGINPERORDER)
cos = make_ib_orders(df_nakeds)

## PLACE THE ORDER

# Archive the orders into `xn_history`

In [ ]:
filename = f"{datetime.now().strftime('%Y%m%d_%I_%M_%p')}_nse_naked_orders.pkl"
pickle_me(ordered, str(ROOT / "data" / "xn_history" / str(filename)))